# Project: Concrete Compressive Strength Prediction
**Course Track:** Machine Learning Internship  
**Task:** Build a regression model to predict the compressive strength of concrete (in MPa) based on its mixtures and age.

---

## 1. Import Libraries and Load Dataset
We start by importing our data manipulation, visualization, and modeling libraries, and reading the raw concrete CSV dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset
df = pd.read_csv('data/concrete.csv')

# Display the first 5 rows
df.head()

## 2. Dataset Inspection
Let's explore the shape, data types, and check for missing values to verify dataset cleanliness.

In [ ]:
# Inspect dimensions and nulls
print("Dataset Shape:", df.shape)
print("\nNull values check:")
print(df.isnull().sum())

# Display summary details
df.info()

## 3. Feature and Target Extraction
We split the dataset into independent features (`X`) containing the concrete ingredients/age, and the continuous dependent target variable (`y`) representing the compressive strength.

In [ ]:
# Separate features (X) and target (y)
X = df.drop(columns=['strength'])
y = df['strength']

print("Features X shape:", X.shape)
print("Target y shape:", y.shape)

## 4. Train-Test Split
We split the data into 80% training data and 20% testing data. We set a fixed seed to ensure output reproducibility.

In [ ]:
# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 5. Modeling Pipeline
We construct a machine learning `Pipeline` combining feature scaling (`StandardScaler`) and a robust non-linear regressor (`RandomForestRegressor`). This setup prevents feature leakage.

In [ ]:
# Setup the modeling pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42))
])

# Fit model to training data
pipe.fit(X_train, y_train)
print("Model pipeline training complete!")

## 6. Model Evaluation & Visualizations
We generate predictions on the test set and evaluate regression metrics (MAE, RMSE, and R² score) followed by prediction visualizations.

In [ ]:
# Generate predictions
y_pred = pipe.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f} MPa")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} MPa")
print(f"R² Score (R-squared): {r2:.4f}")

In [ ]:
# Create regression plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Actual vs Predicted Plot
ax1.scatter(y_test, y_pred, color='blue', alpha=0.6, edgecolors='k')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=3)
ax1.set_xlabel('Actual Compressive Strength (MPa)', fontsize=12)
ax1.set_ylabel('Predicted Compressive Strength (MPa)', fontsize=12)
ax1.set_title('Actual vs. Predicted Concrete Strength', fontsize=14)
ax1.grid(True)

# 2. Residuals Plot
residuals = y_test - y_pred
ax2.scatter(y_pred, residuals, color='purple', alpha=0.6, edgecolors='k')
ax2.axhline(y=0, color='r', linestyle='--', lw=3)
ax2.set_xlabel('Predicted Compressive Strength (MPa)', fontsize=12)
ax2.set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
ax2.set_title('Residuals Analysis', fontsize=14)
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning (Grid Search)
We evaluate alternative configurations via cross-validation to search for optimal estimators.

In [ ]:
# Define parameters using our pipeline step name prefix 'model__'
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(pipe, param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)

# Evaluate tuned model
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print(f"\nTuned Model MAE: {mean_absolute_error(y_test, y_pred_tuned):.2f} MPa")
print(f"Tuned Model R² Score: {r2_score(y_test, y_pred_tuned):.4f}")

## 8. Save Model
We export the best-performing baseline pipeline configuration to a serialized file.

In [ ]:
model_filepath = 'concrete_strength_pipeline.pkl'
with open(model_filepath, 'wb') as file:
    pickle.dump(pipe, file)

print(f"Successfully saved the trained pipeline model to {model_filepath}!")